# ComfyUI — free multi-model image & animation server for BibleMusically

Runs **[ComfyUI](https://github.com/comfyanonymous/ComfyUI)** on a free Kaggle/Colab GPU and exposes its API through a public **cloudflared** tunnel. ComfyUI is the node-based engine that loads many free models and wires them together — the right tool for character consistency, style models, and animation.

**This stack targets your feature list (all free / license-clean for monetization where noted):**
- **Photoreal stills** — SDXL base (and optional FLUX.1 schnell, Apache-2.0)
- **Comic / graphic-novel / style filters** — driven by prompt + LoRA slots you fill from license-clean sources (see notes)
- **Character consistency from example/avatar images** — IP-Adapter (`ComfyUI_IPAdapter_plus`); InstantID installable via ComfyUI-Manager for faces
- **Nuanced variations keeping the character** — IP-Adapter reference + varied prompt/seed/weight
- **Slight animations / music-video motion** — AnimateDiff (`ComfyUI-AnimateDiff-Evolved`) + VideoHelperSuite
- **Add anything else later** — ComfyUI-Manager is installed, so you can add models/nodes from the UI

**Before you run:** enable the GPU — Kaggle: *Settings → Accelerator → GPU T4 x2*; Colab: *Runtime → GPU*.

> **Honest notes.** (1) This is a first-run setup; exact model paths/node versions drift, so if a download 404s, use the ComfyUI-Manager UI to grab the right file — the server still starts. (2) Disk: the full stack is large; toggles below let you skip FLUX to fit a single T4. (3) **Licensing for monetization:** SDXL base (Stability Community License, free under \$1M rev), FLUX.1 *schnell* (Apache-2.0), IP-Adapter & AnimateDiff motion modules (Apache-2.0) are safe. Do **not** drop in FLUX.1 *dev* or random Civitai LoRAs with non-commercial licenses for monetized videos — vet each style model yourself.


## 1. Clone ComfyUI + ComfyUI-Manager


In [ ]:
import os, subprocess, sys

ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
COMFY = os.path.join(ROOT, 'ComfyUI')
# NOTE: enable Internet in Session options first, or these clones fail with
# "Could not resolve host: github.com".
if not os.path.isdir(COMFY):
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git', COMFY], check=True)

# ComfyUI-Manager so you can add models/nodes from the web UI later.
mgr = os.path.join(COMFY, 'custom_nodes', 'ComfyUI-Manager')
if not os.path.isdir(mgr):
    subprocess.run(['git','clone','--depth','1','https://github.com/ltdrdata/ComfyUI-Manager.git', mgr], check=True)

# Installing ComfyUI's requirements may print pip "dependency conflicts" against
# Colab/Kaggle base packages we never use — that noise is non-fatal.
subprocess.run([sys.executable,'-m','pip','install','-q','-r', os.path.join(COMFY,'requirements.txt')], check=True)

# ---- Verify (ComfyUI runs as an app via main.py, not an importable package) ----
ok = os.path.isfile(os.path.join(COMFY,'main.py'))
print(f'  {"✅" if ok else "❌"} ComfyUI at {COMFY} (main.py {"found" if ok else "MISSING"})')
try:
    import torch
    print(f'  ✅ torch        {torch.__version__} (cuda={torch.cuda.is_available()})')
except Exception as ex:
    ok = False
    print(f'  ❌ torch        FAILED: {ex}')
print('\nComfyUI + Manager ready.' if ok else '\n⚠️  Setup incomplete — see above.')


## 2. Install custom nodes for character consistency + animation


In [ ]:
NODES = {
    'ComfyUI_IPAdapter_plus':      'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git',       # character consistency
    'ComfyUI-AnimateDiff-Evolved': 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git',  # motion
    'ComfyUI-VideoHelperSuite':    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',       # video encode
    'comfyui_controlnet_aux':      'https://github.com/Fannovel16/comfyui_controlnet_aux.git',          # pose/composition
}
cn_dir = os.path.join(COMFY,'custom_nodes')
for name, url in NODES.items():
    dst = os.path.join(cn_dir, name)
    if not os.path.isdir(dst):
        subprocess.run(['git','clone','--depth','1',url,dst], check=False)
    req = os.path.join(dst,'requirements.txt')
    if os.path.isfile(req):
        subprocess.run([sys.executable,'-m','pip','install','-q','-r',req], check=False)
print('custom nodes installed:', list(NODES))

## 3. Config toggles

Skip FLUX if you're on a single 16 GB T4 and disk is tight. Set an API key to require `Authorization: Bearer <key>` (must match the app's field).


In [ ]:
# ── Model tier: pick the image-generation stack the server downloads ────────────
# 'quality' : SDXL base + VAE (30 steps, cfg 6.5) — the reliable, license-clean workhorse.
# 'flux'    : also fetch FLUX.1-schnell (Apache-2.0, ~24 GB extra) for top-tier photoreal stills.
#
# Want few-step SPEED? Add a Lightning/Turbo *checkpoint* (not just a LoRA) via ComfyUI-Manager,
# then set it as the checkpoint in the app — the app auto-detects "lightning"/"turbo" in the name
# and drives the correct low steps + cfg for it automatically. A bare LoRA needs a LoraLoader node
# in the workflow to have any effect, so it is intentionally NOT downloaded as a silent default.
MODEL_TIER = 'quality'   # 'quality' | 'flux'

DOWNLOAD_SDXL        = True                 # SDXL base + VAE (the workhorse checkpoint)
DOWNLOAD_IPADAPTER   = True                 # character consistency from example images
DOWNLOAD_ANIMATEDIFF = True                 # slight motion / music-video animation
DOWNLOAD_FLUX        = MODEL_TIER == 'flux' # FLUX.1 schnell (~24 GB) — only on the flux tier
API_KEY = ''                                # match the app's field; blank = open server
print(f'tier={MODEL_TIER} | SDXL={DOWNLOAD_SDXL} IP-Adapter={DOWNLOAD_IPADAPTER} '
      f'AnimateDiff={DOWNLOAD_ANIMATEDIFF} FLUX={DOWNLOAD_FLUX}')

## 4. Download the model stack (license-clean)


In [ ]:
import os, json
from huggingface_hub import hf_hub_download

def M(*p):
    d = os.path.join(COMFY,'models',*p[:-1]); os.makedirs(d, exist_ok=True); return os.path.join(d,p[-1])

# Track what actually landed so the final summary is a real success signal, not silent SKIPs.
_ok, _fail = [], []
def grab(repo, filename, dest_path, repo_type='model', required=True):
    label = os.path.basename(dest_path)
    try:
        if os.path.exists(dest_path) and os.path.getsize(os.path.realpath(dest_path)) > 1_000_000:
            print('  have', label); _ok.append(label); return
        src = hf_hub_download(repo_id=repo, filename=filename, repo_type=repo_type)
        if not os.path.exists(dest_path):
            os.symlink(src, dest_path)
        # Verify: a broken/LFS-pointer download is a few hundred bytes, not a real model.
        if os.path.getsize(os.path.realpath(dest_path)) < 1_000_000:
            raise RuntimeError('downloaded file suspiciously small (LFS pointer / partial?)')
        print('  got', label); _ok.append(label)
    except Exception as ex:
        print('  FAIL', label, '->', ex)
        (_fail if required else _ok).append(label + ('' if required else ' (optional)'))

if DOWNLOAD_SDXL:
    print('SDXL base + VAE...')
    grab('stabilityai/stable-diffusion-xl-base-1.0','sd_xl_base_1.0.safetensors', M('checkpoints','sd_xl_base_1.0.safetensors'))
    grab('madebyollin/sdxl-vae-fp16-fix','sdxl_vae.safetensors', M('vae','sdxl_vae.safetensors'))

if DOWNLOAD_IPADAPTER:
    print('IP-Adapter (character consistency)...')
    grab('h94/IP-Adapter','sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors', M('ipadapter','ip-adapter-plus_sdxl_vit-h.safetensors'))
    grab('h94/IP-Adapter','sdxl_models/ip-adapter_sdxl.safetensors', M('ipadapter','ip-adapter_sdxl.safetensors'))
    grab('h94/IP-Adapter','models/image_encoder/model.safetensors', M('clip_vision','CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors'))

if DOWNLOAD_ANIMATEDIFF:
    print('AnimateDiff SDXL motion module...')
    ad = os.path.join(COMFY,'models','animatediff_models'); os.makedirs(ad, exist_ok=True)
    grab('guoyww/animatediff','mm_sdxl_v10_beta.ckpt', os.path.join(ad,'mm_sdxl_v10_beta.ckpt'), required=False)

if DOWNLOAD_FLUX:
    print('FLUX.1 schnell (photoreal)...')
    grab('black-forest-labs/FLUX.1-schnell','flux1-schnell.safetensors', M('diffusion_models','flux1-schnell.safetensors'))
    grab('black-forest-labs/FLUX.1-schnell','ae.safetensors', M('vae','flux_ae.safetensors'))
    grab('comfyanonymous/flux_text_encoders','clip_l.safetensors', M('text_encoders','clip_l.safetensors'))
    grab('comfyanonymous/flux_text_encoders','t5xxl_fp8_e4m3fn.safetensors', M('text_encoders','t5xxl_fp8_e4m3fn.safetensors'))

# LoRA slot for comic / graphic-novel / art styles (or a Lightning/Turbo LoRA if you wire a
# LoraLoader into a custom workflow) — drop license-clean .safetensors here or use ComfyUI-Manager.
os.makedirs(os.path.join(COMFY,'models','loras'), exist_ok=True)

# ── Write a manifest the app can read to know what this server offers ──────────
# Saved into ComfyUI's web root so it's fetchable at  <tunnel-url>/bm_manifest.json .
# The app primarily discovers checkpoints live via /object_info; this records the intended tier
# and the recommended config so the UI can show what the server is set up for.
manifest = {
    'tier': MODEL_TIER,
    'checkpoints': sorted({n for n in _ok if n.endswith('.safetensors') and ('xl' in n.lower() or 'flux' in n.lower())}),
    'has_ipadapter': DOWNLOAD_IPADAPTER,
    'has_animatediff': DOWNLOAD_ANIMATEDIFF,
    'recommended': {
        'sd_xl_base_1.0.safetensors': {'steps': 30, 'cfg': 6.5, 'sampler': 'dpmpp_2m', 'scheduler': 'karras'},
    },
    'ok': _ok, 'failed': _fail,
}
try:
    web = os.path.join(COMFY, 'web'); os.makedirs(web, exist_ok=True)
    with open(os.path.join(web, 'bm_manifest.json'), 'w') as f: json.dump(manifest, f, indent=2)
    print('wrote web/bm_manifest.json')
except Exception as ex:
    print('could not write manifest:', ex)

print('\n' + '='*60)
print(f'  models ready: {len(_ok)} ok, {len(_fail)} failed')
if _fail:
    print('  MISSING (required):', ', '.join(_fail))
    print('  -> image generation may fail; re-run this cell or use ComfyUI-Manager to fetch them.')
else:
    print('  all required models present. ComfyUI is ready to serve.')
print('='*60)

## 5. Install cloudflared (public tunnel)


In [ ]:
import subprocess
if subprocess.run(['which','cloudflared'], capture_output=True).returncode != 0:
    subprocess.run(
        'curl -L --output /tmp/cloudflared.deb '
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb '
        '&& dpkg -i /tmp/cloudflared.deb', shell=True, check=True)
print('cloudflared ready.')

## 6. Launch ComfyUI + tunnel

Starts ComfyUI on port 8188 and prints the public URL to paste into the app (**Settings → Image engine → ComfyUI**). Keep this cell running.

If you set an API key above, cloudflared can't add auth headers, so protect the endpoint by keeping the URL private (treat it like a password). The app still sends the key for servers that check it.


In [ ]:
import subprocess, sys, os, re, time, threading, urllib.request

PORT = 8188
env = {**os.environ}
if API_KEY:
    env['COMFY_API_KEY'] = API_KEY  # informational; core ComfyUI has no built-in auth

# ── Batch-run guard (FIRST — before launching ComfyUI) ─────────────
# A GPU-off batch run is a cheap source-update push. ComfyUI's main.py aborts with
# 'Torch not compiled with CUDA enabled' if started without a GPU, so we must decide
# BEFORE launching it: GPU-off batch -> skip everything; GPU batch (the app's Start
# server) -> launch ComfyUI and serve.
_is_batch = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive').lower() == 'batch'

# Two independent questions, asked separately because they fail apart — and which one failed IS
# the diagnosis:
#   nvidia-smi  — does this container have a GPU and a working driver at all?
#   torch.cuda  — can the framework that runs the model actually reach it?
# A GPU-off source-update push answers no to both, and so does Kaggle declining an accelerator.
# An install step that replaced Kaggle's CUDA torch with a CPU-only wheel answers YES to the
# first and no to the second. The old guard asked only nvidia-smi and reported every "no" as an
# exhausted weekly quota — which sent people to a quota page that had 29.8 of 30 hours left on it.
_smi_rc, _smi_note = None, ''
try:
    _smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=120)
    _smi_rc = _smi.returncode
    _smi_note = (_smi.stderr or '').strip().replace('\n', ' ')[:200]
except FileNotFoundError:
    _smi_note = 'nvidia-smi is not installed on this container'
except Exception as _ex:
    _smi_note = '{}: {}'.format(type(_ex).__name__, _ex)
try:
    import torch as _t
    _torch_cuda, _torch_ver = bool(_t.cuda.is_available()), _t.__version__
except Exception as _ex:
    _torch_cuda, _torch_ver = False, 'unavailable ({})'.format(type(_ex).__name__)

# Serving is gated on torch rather than on the driver, because the model is loaded onto whatever
# device torch reports. A container that has a GPU torch cannot see would otherwise open a public
# tunnel to a server generating on CPU — minutes of audio at hours of wall clock, which is a worse
# outcome than not starting, and much harder to diagnose from the app.
_has_gpu = _torch_cuda
if _is_batch and not _has_gpu:
    print('=' * 70)
    print('  NO GPU ON THIS RUN — not serving.')
    print('  nvidia-smi: ' + ('exit {}'.format(_smi_rc) if _smi_rc is not None else 'did not run')
          + (' — {}'.format(_smi_note) if _smi_note else ''))
    print('  torch {}: cuda.is_available() = {}'.format(_torch_ver, _torch_cuda))
    print('  CUDA_VISIBLE_DEVICES = {!r}'.format(os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>')))
    if _smi_rc == 0:
        print('  GPU PRESENT BUT TORCH CANNOT USE IT — an install step in this notebook replaced')
        print("  Kaggle's CUDA build of torch with a CPU-only one. Fix that cell; the quota is")
        print('  not the problem here.')
    else:
        print('  KAGGLE GAVE THIS SESSION NO ACCELERATOR. The app always asks for one, so this is')
        print('  the scheduler declining: the weekly quota is spent, both GPU session slots are')
        print('  busy, or no GPU was free at that moment. That last case is common and transient —')
        print('  if the quota page still shows hours left, simply start again.')
        print('  Quota: https://www.kaggle.com/settings  (Accelerator usage).')
    print('  (A deliberate GPU-off push is just a cheap source update - nothing is wrong.)')
    print('=' * 70)
    print('Start the server from the app (Start server button) or run interactively with GPU on.')
else:
    args = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT), '--preview-method', 'auto']
    comfy = subprocess.Popen(args, cwd=COMFY, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)

    def _pump(p, tag):
        for line in p.stdout: print(f'[{tag}] {line}', end='')
    threading.Thread(target=_pump, args=(comfy,'comfy'), daemon=True).start()

    print('Waiting for ComfyUI to come up (first start loads nodes)...')
    for _ in range(90):
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=3); print('ComfyUI is up.'); break
        except Exception:
            time.sleep(3)
    else:
        print('WARNING: ComfyUI did not respond; check [comfy] logs above.')

    import urllib.request, urllib.error

    # ── Reliable public tunnel with self-healing ───────────────────────────────
    # The failure this fixes: cloudflared prints a *.trycloudflare.com URL and even registers an
    # edge connection, yet the Cloudflare edge never actually ROUTES the hostname, so the URL never
    # answers and the app times out. Quick tunnels are flaky per-process and QUIC (UDP) egress can
    # be throttled. So we: (1) probe our OWN public URL to confirm it truly routes, (2) auto-restart
    # cloudflared — first over QUIC, then over HTTP/2, which survives UDP throttling — and (3) fall
    # back to localhost.run (ssh) if cloudflared keeps failing. Only a URL that actually ANSWERS is
    # printed as ready, and every step is logged so a failure is diagnosable from the app's log tail.
    _url_re = re.compile(r'https://[-a-z0-9]+\.(?:trycloudflare\.com|lhr\.life|serveo\.net)')

    def _probe_public(url, timeout=8):
        # True iff the tunnel truly routes: ANY HTTP status < 500 back proves the edge reached our
        # server. A connection error/timeout, or Cloudflare's own 5xx (e.g. 530 = tunnel down),
        # means "not routed yet".
        try:
            with urllib.request.urlopen(url.rstrip('/') + '/', timeout=timeout) as r:
                return r.status < 500
        except urllib.error.HTTPError as he:
            return he.code < 500
        except Exception:
            return False

    def _pump(proc, holder, tag='tunnel'):
        def _run():
            for line in proc.stdout:
                print(f'[{tag}] {line}', end='')
                m = _url_re.search(line)
                if m and not holder.get('url'):
                    holder['url'] = m.group(0)
        threading.Thread(target=_run, daemon=True).start()

    def _spawn_cloudflared(protocol):
        args = ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}']
        if protocol:
            args += ['--protocol', protocol]
        print(f'[tunnel] launching cloudflared (protocol={protocol or "auto"})...', flush=True)
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _spawn_localhostrun():
        print('[tunnel] launching localhost.run over ssh...', flush=True)
        p = subprocess.Popen(
            ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'UserKnownHostsFile=/dev/null',
             '-o', 'ServerAliveInterval=30', '-R', f'80:localhost:{PORT}', 'nokey@localhost.run'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _bring_up(proc, holder, url_wait=30, route_wait=75):
        t0 = time.time()
        while time.time() - t0 < url_wait and not holder.get('url'):
            if proc.poll() is not None:
                print('[tunnel] process exited before printing a URL.', flush=True); return None
            time.sleep(1)
        url = holder.get('url')
        if not url:
            print(f'[tunnel] no URL within {url_wait}s.', flush=True); return None
        print(f'Tunnel URL: {url}', flush=True)
        print('Waiting for the edge to route it...', flush=True)
        t1 = time.time()
        while time.time() - t1 < route_wait:
            if proc.poll() is not None:
                print('[tunnel] tunnel process exited during the routing wait.', flush=True); return None
            if _probe_public(url):
                print(f'[tunnel] OK routed and answering after {int(time.time()-t1)}s.', flush=True)
                return url
            time.sleep(4)
        print(f'[tunnel] {url} never answered within {route_wait}s - treating as dead.', flush=True)
        return None

    _attempts = [('cf', 'quic'), ('cf', 'http2'), ('lhr', None)]
    active_proc = None; public_url = None
    for _i, (_kind, _proto) in enumerate(_attempts, 1):
        print(f'\n[tunnel] ===== attempt {_i}/{len(_attempts)}: {_kind} {_proto or ""} =====', flush=True)
        try:
            _p, _h = _spawn_cloudflared(_proto) if _kind == 'cf' else _spawn_localhostrun()
        except FileNotFoundError as _e:
            print(f'[tunnel] cannot launch ({_e}); skipping this attempt.', flush=True); continue
        _routed = _bring_up(_p, _h)
        if _routed:
            active_proc, public_url = _p, _routed; break
        try: _p.terminate()
        except Exception: pass
        time.sleep(2)

    print('\n' + '=' * 70)
    if public_url:
        print('  PASTE THIS INTO THE APP  ->  Settings -> ComfyUI server URL:')
        print(f'  {public_url}')
    else:
        print('  FAILED: no working public tunnel after all attempts. The local server is fine,')
        print('  but nothing outside can reach it - retry "Start & connect" from the app.')
    print('=' * 70, flush=True)

    if public_url and active_proc:
        # ── Idle-shutdown watchdog ──────────────────────────────────────
        # After IDLE_SHUTDOWN_MIN minutes with no ESTABLISHED connection to the server port, stop the
        # tunnel so a forgotten run stops burning GPU quota. App polling / liveness counts as activity.
        IDLE_SHUTDOWN_MIN = 15
        def _idle_watchdog():
            _port_hex = ':%04X' % PORT
            _last = time.time()
            while True:
                time.sleep(30)
                _active = False
                for _tbl in ('/proc/net/tcp', '/proc/net/tcp6'):
                    try:
                        with open(_tbl) as _f:
                            for _l in _f.readlines()[1:]:
                                _q = _l.split()
                                if _q[1].endswith(_port_hex) and _q[3] == '01':
                                    _active = True; break
                    except OSError:
                        _active = True
                    if _active: break
                if _active:
                    _last = time.time()
                elif time.time() - _last > IDLE_SHUTDOWN_MIN * 60:
                    print(f'[watchdog] No requests for {IDLE_SHUTDOWN_MIN} min - shutting down to save GPU quota.', flush=True)
                    try: active_proc.terminate()
                    except Exception: pass
                    return
        threading.Thread(target=_idle_watchdog, daemon=True).start()
        print(f'Keep this cell running. Idle watchdog armed: auto-stops after {IDLE_SHUTDOWN_MIN} min idle.', flush=True)
        active_proc.wait()